In [1]:
!pip install -q unsloth 
!pip install --upgrade -q transformers==4.53.3
!pip install --no-deps trl==0.22.2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.9/337.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.1/888.1 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch

In [ ]:
import transformers
from transformers import AutoProcessor, AutoModelForCausalLM  
from PIL import Image
import requests
import copy
%matplotlib inline  
print(transformers.__version__)

In [ ]:
import gc
llm_model=None
processor=None
llm_tokenizer=None
gc.collect()
torch.cuda.empty_cache()


In [ ]:
model_id = 'microsoft/Florence-2-large-ft'
vision_model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, torch_dtype='auto').eval().cuda()
vision_processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

In [ ]:
def run_example(task_prompt, text_input=None):
    if text_input is None:
        prompt = task_prompt
    else:
        prompt = task_prompt + text_input
    inputs = vision_processor(text=prompt, images=image, return_tensors="pt").to('cuda', torch.float16)
    generated_ids = vision_model.generate(
      input_ids=inputs["input_ids"].cuda(),
      pixel_values=inputs["pixel_values"].cuda(),
      max_new_tokens=1024,
      early_stopping=False,
      do_sample=False,
      num_beams=3,
    )
    generated_text = vision_processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer =vision_processor.post_process_generation(
        generated_text, 
        task=task_prompt, 
        image_size=(image.width, image.height)
    )

    return parsed_answer

In [ ]:
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg?download=true"
image = Image.open(requests.get(url, stream=True).raw)
task_prompt = '<CAPTION>'
run_example(task_prompt)

In [ ]:
# --- Шаг 2: Модификация вызова для получения векторов (с исправлением) ---

def get_vectors_from_florence2(model, processor, image, task_prompt):
    """
    Функция для извлечения векторов (скрытых состояний энкодера) из Florence-2.
    """
    # 1. Подготовка входов
    inputs = processor(
        text=task_prompt, 
        images=image, 
        return_tensors="pt"
    ).to('cuda', torch.float16)

    print("Входные данные подготовлены. Shapes и dtypes:")
    print(f"  - input_ids: {inputs['input_ids'].shape}, dtype: {inputs['input_ids'].dtype}")
    print(f"  - pixel_values: {inputs['pixel_values'].shape}, dtype: {inputs['pixel_values'].dtype}")
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1,
            output_hidden_states=True,
            return_dict_in_generate=True
        )

    encoder_last_hidden_state = outputs.encoder_hidden_states[-1]
    
    return encoder_last_hidden_state



image_vectors = get_vectors_from_florence2(vision_model, vision_processor, image, task_prompt)


print(f"Тип полученного объекта: {type(image_vectors)}")
print(f"Shape тензора: {image_vectors.shape}")
print(f"Тип данных (dtype): {image_vectors.dtype}")
print(f"Устройство: {image_vectors.device}")

batch_size, sequence_length, hidden_size = image_vectors.shape
print(f"\nИнтерпретация размерности [Batch, Sequence, Hidden]:")
print(f"  - Batch Size: {batch_size}")
print(f"  - Sequence Length (визуальные токены): {sequence_length}")
print(f"  - Hidden Size (размерность вектора): {hidden_size}")

    


In [ ]:
vision_model.vision_tower.config

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image
import requests
from types import SimpleNamespace


class FlorenceVisionTower(nn.Module):
    def __init__(self, vision_tower, args, delay_load=False):
        super().__init__()

        self.is_loaded = False
        self.vision_tower_name = vision_tower

        # В нашем примере мы всегда загружаем модель сразу
        self.load_model()


    def load_model(self, device_map=None):
        if self.is_loaded:
            print('{} is already loaded, `load_model` called again, skipping.'.format(self.vision_tower_name))
            return
        
        self.device_ = "cuda" if torch.cuda.is_available() else "cpu"
        self.dtype_ = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

        print(f"Loading model on {self.device_} with dtype {self.dtype_}...")

        self.image_processor = AutoProcessor.from_pretrained(self.vision_tower_name, trust_remote_code=True)
        self.vision_tower = AutoModelForCausalLM.from_pretrained(
            self.vision_tower_name, 
            trust_remote_code=True
        ).to(device=self.device_, dtype=self.dtype_).eval()
        
        self.vision_tower.requires_grad_(False)
        self.is_loaded = True


    @torch.no_grad()
    def forward(self, images):
        task_ids = torch.tensor([
            [0, 47066, 21700, 11, 4617, 99, 16, 2343, 11, 5, 2274, 4, 2, 1],
            [0, 2264, 16, 5, 2788, 11, 5, 2274, 116, 2, 1, 1, 1, 1],
            [0, 574, 22486, 5, 8720, 11, 5, 2274, 6, 19, 49, 24173, 4, 2]
        ]).to(device=self.device)

        generated_ids, image_feature, encoder_last_hidden_state = self.vision_tower.generate(
            input_ids=task_ids,
            pixel_values=images,
            max_new_tokens=1,
            do_sample=False,
            num_beams=1,
        )
        print(self.vision_tower.config)
    return image_feature, encoder_last_hidden_state

    @property
    def dtype(self):
        return self.dtype_

    @property
    def device(self):
        return self.device_
    
    @property
    def config(self):
        return self.vision_tower.config

    @property
    def hidden_size(self):
        return self.config.hidden_size


if __name__ == "__main__":
    
    vision_tower_name = 'microsoft/Florence-2-large-ft'
    vision_tower_component = FlorenceVisionTower(vision_tower_name, mock_args)
    
    url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg?download=true"
    image = Image.open(requests.get(url, stream=True).raw)
    
    # Используем процессор, который уже является частью нашего компонента
    image_inputs = vision_tower_component.image_processor.image_processor(images=image, return_tensors="pt")
    pixel_values = image_inputs.pixel_values.to(
        device=vision_tower_component.device, 
        dtype=vision_tower_component.dtype
    )
    
        
    print("\nПодготовлены входные данные:")
    print(f"  - pixel_values_expanded.shape: {pixel_values_expanded.shape}")
    
    print("Извлечение признаков...")
    image_feature, encoder_hidden_state = vision_tower_component(pixel_values_expanded)
    
    print("\nМультизадачные признаки успешно извлечены!")

    print("\n--- Анализ 'image_feature' ---")
    print(f"Shape тензора: {image_feature.shape}")
    print(f"Интерпретация: [Кол-во задач, 1, Размерность]")

    print("\n--- Анализ 'encoder_last_hidden_state' ---")
    print(f"Shape тензора: {encoder_hidden_state.shape}")
    print(f"Интерпретация: [Кол-во задач, Кол-во патчей, Размерность]")

In [ ]:
import gc
llm_model=None
processor=None
model=None
llm_tokenizer=None
gc.collect()
torch.cuda.empty_cache()


In [ ]:
gc.collect()
torch.cuda.empty_cache()
image_vectors=None

In [ ]:
inputs = vision_processor(
        text=task_prompt, 
        images=image, 
        return_tensors="pt"
    ).to('cuda', torch.float16)

outputs = vision_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1,
            output_hidden_states=True,
            return_dict_in_generate=True
        )


In [ ]:
llm_model, llm_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

In [ ]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = llm_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = llm_tokenizer([text], return_tensors="pt").to(llm_model.device)
generated_ids = llm_model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = llm_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
response

In [ ]:
llm_model = FastLanguageModel.get_peft_model(
    llm_model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

In [ ]:
for module in vision_model.children():
    print(module)
    break

In [ ]:
def get_features_from_davit(model, processor, image):
    """
    Напрямую извлекает визуальные признаки из Vision Tower (DaViT).
    """

    
    # 2. Используем процессор, чтобы подготовить ТОЛЬКО изображение.
    # Текст нам для этой операции не нужен.
    # Важно: `do_resize=True, size={"longest_edge": 768}` - это параметры по умолчанию, 
    # которые использует процессор Florence-2 для изображений. Указываем их для ясности.
    image_inputs = vision_processor(
        text=task_prompt, 
        images=image, 
        return_tensors="pt"
    ).to('cuda', torch.float16)

    
    # 3. Приводим тензор изображения к нужному типу и устройству
    pixel_values = image_inputs.pixel_values.to(device=device, )
    
    with torch.no_grad():
        # Важно: мы не используем `output_hidden_states` и `return_dict...`,
        # так как модель Florence-2 переопределяет `generate` и возвращает кортеж напрямую.
        generated_ids, image_feature, encoder_last_hidden_state = model.vision_tower.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1,   # Нам не нужна генерация, только запуск энкодера
            num_beams=1,        # Простой проход, без сложного поиска
            do_sample=False     # Детерминированный результат
        )
        
    return image_feature, encoder_last_hidden_state



image_feature, encoder_hidden_state = get_features_from_davit(vision_model, vision_processor, image)

print("\nМультизадачные признаки успешно извлечены!")

print("\n--- Анализ 'image_feature' ---")
print(f"Shape тензора: {image_feature.shape}")
print(f"Тип данных (dtype): {image_feature.dtype}")

print("\n--- Анализ 'encoder_last_hidden_state' ---")
print(f"Shape тензора: {encoder_hidden_state.shape}")
print(f"Тип данных (dtype): {encoder_hidden_state.dtype}")


In [ ]:
CONTROLLER_HEART_BEAT_EXPIRATION = 30
WORKER_HEART_BEAT_INTERVAL = 15

LOGDIR = "."

IGNORE_INDEX = -100
IMAGE_TOKEN_INDEX = -200
DEFAULT_IMAGE_TOKEN = "<image>"
DEFAULT_IMAGE_PATCH_TOKEN = "<im_patch>"
DEFAULT_IM_START_TOKEN = "<im_start>"
DEFAULT_IM_END_TOKEN = "<im_end>"
IMAGE_PLACEHOLDER = "<image-placeholder>"

In [ ]:
from PIL import Image
from io import BytesIO
import base64
import torch
import math
import ast

from transformers import StoppingCriteria


def select_best_resolution(original_size, possible_resolutions):
    """
    Selects the best resolution from a list of possible resolutions based on the original size.

    Args:
        original_size (tuple): The original size of the image in the format (width, height).
        possible_resolutions (list): A list of possible resolutions in the format [(width1, height1), (width2, height2), ...].

    Returns:
        tuple: The best fit resolution in the format (width, height).
    """
    original_width, original_height = original_size
    best_fit = None
    max_effective_resolution = 0
    min_wasted_resolution = float('inf')

    for width, height in possible_resolutions:
        scale = min(width / original_width, height / original_height)
        downscaled_width, downscaled_height = int(original_width * scale), int(original_height * scale)
        effective_resolution = min(downscaled_width * downscaled_height, original_width * original_height)
        wasted_resolution = (width * height) - effective_resolution

        if effective_resolution > max_effective_resolution or (effective_resolution == max_effective_resolution and wasted_resolution < min_wasted_resolution):
            max_effective_resolution = effective_resolution
            min_wasted_resolution = wasted_resolution
            best_fit = (width, height)

    return best_fit


def resize_and_pad_image(image, target_resolution):
    """
    Resize and pad an image to a target resolution while maintaining aspect ratio.

    Args:
        image (PIL.Image.Image): The input image.
        target_resolution (tuple): The target resolution (width, height) of the image.

    Returns:
        PIL.Image.Image: The resized and padded image.
    """
    original_width, original_height = image.size
    target_width, target_height = target_resolution

    scale_w = target_width / original_width
    scale_h = target_height / original_height

    if scale_w < scale_h:
        new_width = target_width
        new_height = min(math.ceil(original_height * scale_w), target_height)
    else:
        new_height = target_height
        new_width = min(math.ceil(original_width * scale_h), target_width)

    # Resize the image
    resized_image = image.resize((new_width, new_height))

    new_image = Image.new('RGB', (target_width, target_height), (0, 0, 0))
    paste_x = (target_width - new_width) // 2
    paste_y = (target_height - new_height) // 2
    new_image.paste(resized_image, (paste_x, paste_y))

    return new_image


def divide_to_patches(image, patch_size):
    """
    Divides an image into patches of a specified size.

    Args:
        image (PIL.Image.Image): The input image.
        patch_size (int): The size of each patch.

    Returns:
        list: A list of PIL.Image.Image objects representing the patches.
    """
    patches = []
    width, height = image.size
    for i in range(0, height, patch_size):
        for j in range(0, width, patch_size):
            box = (j, i, j + patch_size, i + patch_size)
            patch = image.crop(box)
            patches.append(patch)

    return patches


def get_anyres_image_grid_shape(image_size, grid_pinpoints, patch_size):
    """
    Calculate the shape of the image patch grid after the preprocessing for images of any resolution.

    Args:
        image_size (tuple): The size of the input image in the format (width, height).
        grid_pinpoints (str): A string representation of a list of possible resolutions.
        patch_size (int): The size of each image patch.

    Returns:
        tuple: The shape of the image patch grid in the format (width, height).
    """
    if type(grid_pinpoints) is list:
        possible_resolutions = grid_pinpoints
    else:
        possible_resolutions = ast.literal_eval(grid_pinpoints)
    width, height = select_best_resolution(image_size, possible_resolutions)
    return width // patch_size, height // patch_size


def process_anyres_image(image, processor, grid_pinpoints):
    """
    Process an image with variable resolutions.

    Args:
        image (PIL.Image.Image): The input image to be processed.
        processor: The image processor object.
        grid_pinpoints (str): A string representation of a list of possible resolutions.

    Returns:
        torch.Tensor: A tensor containing the processed image patches.
    """
    if type(grid_pinpoints) is list:
        possible_resolutions = grid_pinpoints
    else:
        possible_resolutions = ast.literal_eval(grid_pinpoints)
    best_resolution = select_best_resolution(image.size, possible_resolutions)
    image_padded = resize_and_pad_image(image, best_resolution)

    patches = divide_to_patches(image_padded, processor.crop_size['height'])

    image_original_resize = image.resize((processor.size['shortest_edge'], processor.size['shortest_edge']))

    image_patches = [image_original_resize] + patches
    image_patches = [processor.preprocess(image_patch, return_tensors='pt')['pixel_values'][0]
                     for image_patch in image_patches]
    return torch.stack(image_patches, dim=0)


def load_image_from_base64(image):
    return Image.open(BytesIO(base64.b64decode(image)))


def expand2square(pil_img, background_color):
    width, height = pil_img.size
    if width == height:
        return pil_img
    elif width > height:
        result = Image.new(pil_img.mode, (width, width), background_color)
        result.paste(pil_img, (0, (width - height) // 2))
        return result
    else:
        result = Image.new(pil_img.mode, (height, height), background_color)
        result.paste(pil_img, ((height - width) // 2, 0))
        return result


def process_images(images, image_processor, model_cfg):
    image_aspect_ratio = getattr(model_cfg, "image_aspect_ratio", None)
    new_images = []
    # task = [
    #     'What is the text in the image?',
    #     'What is the text in the image, with regions?',
    #     'What does the image describe?',
    #     'Describe in detail what is shown in the image.',
    #     'Describe with a paragraph what is shown in the image.',
    #     'Locate the objects with category name in the image.',
    #     'Locate the objects in the image, with their descriptions.',
    #     'Locate the region proposals in the image.'
    # ]
    
    task = [
        'Describe in detail what is shown in the image.',
        'What is the text in the image?',
        'Locate the objects in the image, with their descriptions.'
    ]

    if image_aspect_ratio == 'pad':
        for image in images:
            image = expand2square(image, tuple(int(x*255) for x in image_processor.image_mean))
            image = image_processor(text=task, images=image, return_tensors='pt', image_mean=image_processor.image_mean, image_std=image_processor.image_std, padding=True)['pixel_values'][0]
            new_images.append(image)
    elif image_aspect_ratio == "anyres":
        for image in images:
            image = process_anyres_image(image, image_processor, model_cfg.image_grid_pinpoints)
            new_images.append(image)
    else:
        return image_processor(images, return_tensors='pt')['pixel_values']
    if all(x.shape == new_images[0].shape for x in new_images):
        new_images = torch.stack(new_images, dim=0)
    return new_images


def tokenizer_image_token(prompt, tokenizer, image_token_index=IMAGE_TOKEN_INDEX, return_tensors=None):
    prompt_chunks = [tokenizer(chunk).input_ids for chunk in prompt.split('<image>')]

    def insert_separator(X, sep):
        return [ele for sublist in zip(X, [sep]*len(X)) for ele in sublist][:-1]

    input_ids = []
    offset = 0
    if len(prompt_chunks) > 0 and len(prompt_chunks[0]) > 0 and prompt_chunks[0][0] == tokenizer.bos_token_id:
        offset = 1
        input_ids.append(prompt_chunks[0][0])

    for x in insert_separator(prompt_chunks, [image_token_index] * (offset + 1)):
        input_ids.extend(x[offset:])

    if return_tensors is not None:
        if return_tensors == 'pt':
            return torch.tensor(input_ids, dtype=torch.long)
        raise ValueError(f'Unsupported tensor type: {return_tensors}')
    return input_ids


def get_model_name_from_path(model_path):
    model_path = model_path.strip("/")
    model_paths = model_path.split("/")
    if model_paths[-1].startswith('checkpoint-'):
        return model_paths[-2] + "_" + model_paths[-1]
    else:
        return model_paths[-1]

class KeywordsStoppingCriteria(StoppingCriteria):
    def __init__(self, keywords, tokenizer, input_ids):
        self.keywords = keywords
        self.keyword_ids = []
        self.max_keyword_len = 0
        for keyword in keywords:
            cur_keyword_ids = tokenizer(keyword).input_ids
            if len(cur_keyword_ids) > 1 and cur_keyword_ids[0] == tokenizer.bos_token_id:
                cur_keyword_ids = cur_keyword_ids[1:]
            if len(cur_keyword_ids) > self.max_keyword_len:
                self.max_keyword_len = len(cur_keyword_ids)
            self.keyword_ids.append(torch.tensor(cur_keyword_ids))
        self.tokenizer = tokenizer
        self.start_len = input_ids.shape[1]
    
    def call_for_batch(self, output_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        offset = min(output_ids.shape[1] - self.start_len, self.max_keyword_len)
        self.keyword_ids = [keyword_id.to(output_ids.device) for keyword_id in self.keyword_ids]
        for keyword_id in self.keyword_ids:
            truncated_output_ids = output_ids[0, -keyword_id.shape[0]:]
            if torch.equal(truncated_output_ids, keyword_id):
                return True
        outputs = self.tokenizer.batch_decode(output_ids[:, -offset:], skip_special_tokens=True)[0]
        for keyword in self.keywords:
            if keyword in outputs:
                return True
        return False
    
    def __call__(self, output_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        outputs = []
        for i in range(output_ids.shape[0]):
            outputs.append(self.call_for_batch(output_ids[i].unsqueeze(0), scores))
        return all(outputs)

In [ ]:
import torch
import torch.nn as nn

from transformers import AutoProcessor, AutoModelForCausalLM 

class FlorenceVisionTower(nn.Module):
    def __init__(self, vision_tower, args, delay_load=False):
        super().__init__()

        self.is_loaded = False
        self.vision_tower_name = vision_tower

        if not delay_load:
            self.load_model()
        elif getattr(args, 'unfreeze_mm_vision_tower', False):
            self.load_model()
        else:
            self.load_model()


    def load_model(self, device_map=None):
        if self.is_loaded:
            print('{} is already loaded, `load_model` called again, skipping.'.format(self.vision_tower_name))
            return

        self.image_processor = AutoProcessor.from_pretrained(self.vision_tower_name, trust_remote_code=True)
        self.vision_tower = AutoModelForCausalLM.from_pretrained(self.vision_tower_name, trust_remote_code=True).to(torch.bfloat16)
        self.vision_tower.requires_grad_(False)

        self.is_loaded = True


    @torch.no_grad()
    def forward(self, images):
        
        
        # task = [
        #     'Describe in detail what is shown in the image.',
        #     'What is the text in the image?',
        #     'Locate the objects in the image, with their descriptions.',
        # ]

        ## token id for three tasks prompt
        task_ids = torch.tensor([
            [0, 47066, 21700, 11, 4617, 99, 16, 2343, 11, 5, 2274, 4, 2, 1],
            [0, 2264, 16, 5, 2788, 11, 5, 2274, 116, 2, 1, 1, 1, 1],
            [0, 574, 22486, 5, 8720, 11, 5, 2274, 6, 19, 49, 24173, 4, 2]
        ]).to(device=self.device)


        with torch.no_grad():
            generated_ids, image_feature, encoder_last_hidden_state = self.vision_tower.generate(
                input_ids=task_ids,
                pixel_values=images,
                max_new_tokens=1,
                do_sample=False,
                num_beams=1,
            )
            return image_feature, encoder_last_hidden_state


    @property
    def dummy_feature(self):
        return torch.zeros(1, self.hidden_size, device=self.device, dtype=self.dtype)

    @property
    def dtype(self):
        return self.vision_tower.dtype

    @property
    def device(self):
        return self.vision_tower.device

    @property
    def config(self):
        if self.is_loaded:
            return self.vision_tower.config
        else:
            return self.cfg_only

    @property
    def hidden_size(self):
        return self.config.hidden_size

    @property
    def num_patches_per_side(self):
        return self.config.image_size // self.config.patch_size

    @property
    def num_patches(self):
        return (self.config.image_size // self.config.patch_size) ** 2

In [ ]:
  # Initialize the FlorenceVisionTower  
class Args:  
    unfreeze_mm_vision_tower = False  
    image_aspect_ratio = 'pad'  # or None, depending on your needs  
  
args = Args()  
vision_tower_name = 'microsoft/Florence-2-large-ft' # or your Florence model path  
  
# Create and load the vision tower  
vision_tower = FlorenceVisionTower(vision_tower_name, args)  
  
# Load an image  
image_path = "path/to/your/image.jpg"  
image = Image.open(image_path).convert('RGB')  
  
# Preprocess the image  
images = [image]  
pixel_values = process_images(images, vision_tower.image_processor, args)  
##ВОТ ТУТ ЗАМЕНИТЬ НА ТО ЧТО ВЫШЕ БЫЛО!!!!
  
# Move to the same device as the vision tower  
pixel_values = pixel_values.to(device=vision_tower.device, dtype=vision_tower.dtype)  
  
# Perform forward pass  
image_feature, encoder_last_hidden_state = vision_tower(pixel_values)  
  
# Now you have both embeddings:  